# Evaluation
## Testing datasets
### Retrieval ground truth

In [1]:
from wikifin_rag.ingest import load_wikifin_data

In [2]:
documents = load_wikifin_data()

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [4]:
data_gen_instructions = """
You emulate a student or young professional who has questions about personal finance.
Formulate {} questions this person might ask based on a document passage.
The passage should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from wikifin_rag.evaluation_utils import llm_structured_retry

In [6]:
load_dotenv(override=True)
openai_client = OpenAI()

In [7]:
def generate_document_ground_truth(doc, n=5):
    user_prompt = doc['content']

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions.format(n),
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [8]:
from concurrent.futures import ThreadPoolExecutor
from wikifin_rag.evaluation_utils import map_progress, calculate_total_cost
import pandas as pd
from wikifin_rag.config import PROJECT_ROOT
from pathlib import Path
import numpy as np
import os

In [9]:
def generate_corpus_ground_truth(documents, n=5, file_path=PROJECT_ROOT / "data" / "evals" / "ground_truth-new.csv"):
    with ThreadPoolExecutor(max_workers=6) as pool:
        results = map_progress(pool, documents, lambda doc: generate_document_ground_truth(doc, n=n))

    ground_truth = []
    usages = []

    for records, usage in results:
        ground_truth.extend(records)
        usages.append(usage)

    # total_cost = calculate_total_cost(usages)

    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df_ground_truth = pd.DataFrame(ground_truth)
    df_ground_truth.to_csv(file_path, index=False)


In [10]:
# number of documents to use
n_documents = 300

# number of question to generate per document
n_questions = 5

# file path for the generated ground truth dataset
ground_truth_file_path = PROJECT_ROOT / "data" / "evals" / "ground_truth-new.csv"

In [11]:
# TODO: change to True to force a ground truth data refresh
refresh_ground_truth = False

In [12]:
ground_truth_docs = np.random.choice(documents, size=n_documents, replace=False)

In [13]:
if refresh_ground_truth or not os.path.exists(ground_truth_file_path):
    generate_corpus_ground_truth(documents=ground_truth_docs, n=n_questions, file_path=ground_truth_file_path)

  0%|          | 0/300 [00:00<?, ?it/s]

## Retrieval function evaluation
### Text Search

In [14]:
from wikifin_rag.evaluation_utils import evaluate
from wikifin_rag.ingest import build_text_index

In [15]:
ts_index = build_text_index(documents=documents)

In [16]:
def text_search(query, boost_dict=None, num_results=5):
    return ts_index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

In [17]:
df_ground_truth = pd.read_csv(ground_truth_file_path)
ground_truth = df_ground_truth.to_dict(orient="records")

In [18]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import pickle

In [19]:
def ts_objective(params):
    metrics = evaluate(
        ground_truth=ground_truth,
        search_function=lambda query: text_search(query=query, boost_dict=params)
    )

    return {
        'loss': -round(metrics['mrr'], 4),
        'status': STATUS_OK,
        'params': params,
        'metrics': metrics
    }

In [20]:
n_evals = 25

In [21]:
ts_trials_file_path = PROJECT_ROOT / "data" / "evals" / "ts_trials.pkl"

In [22]:
# TODO: change to True to force a ground truth data refresh
refresh_search_optimization = False

In [23]:
ts_search_space = {
    "title": hp.uniform("title", 0, 20),
    "section": hp.uniform("section", 0, 20),
    "content": hp.uniform("content", 0, 20)
}

In [ ]:
if refresh_search_optimization or not os.path.exists(ts_trials_file_path):
    ts_trials_file_path.parent.mkdir(parents=True, exist_ok=True)
    
    trials = Trials()

    best_params = fmin(
        fn=ts_objective,
        space=ts_search_space,
        algo=tpe.suggest,
        max_evals=n_evals,
        trials=trials
    )

    trials_data = [{
        'id': trial['tid'],
        'mrr': trial['result']['metrics']['mrr'],
        'hit_rate': trial['result']['metrics']['hit_rate'],
        'params': trial['result']['params'],
    } for trial in trials.trials]
    
    trials_data_df = pd.DataFrame(trials_data)

    with open(ts_trials_file_path, "wb") as file:
        pickle.dump(trials_data, file)

  0%|          | 0/25 [00:00<?, ?trial/s, best loss=?]

  0%|          | 0/1500 [00:00<?, ?it/s]

  4%|▍         | 1/25 [00:07<02:50,  7.10s/trial, best loss: -0.4458]

  0%|          | 0/1500 [00:00<?, ?it/s]

  8%|▊         | 2/25 [00:14<02:41,  7.04s/trial, best loss: -0.4458]

...

 92%|█████████▏| 23/25 [02:52<00:15,  7.93s/trial, best loss: -0.4458]

  0%|          | 0/1500 [00:00<?, ?it/s]

 96%|█████████▌| 24/25 [02:59<00:07,  7.93s/trial, best loss: -0.4458]

  0%|          | 0/1500 [00:00<?, ?it/s]

100%|██████████| 25/25 [03:08<00:00,  7.52s/trial, best loss: -0.4458]


In [65]:
with open(ts_trials_file_path, "rb") as file:
    ts_trials_data = pickle.load(file)

In [66]:
ts_best_metrics = max(ts_trials_data, key=lambda d: d['mrr'])

In [68]:
print(f"Text search MRR: {round(ts_best_metrics['mrr'], 4)} | Hit Rate: {round(100 * ts_best_metrics['hit_rate'], 2)}%")

Text search MRR: 0.4458 | Hit Rate: 57.07%


### Vector Search

In [25]:
from wikifin_rag.ingest import build_vector_index
from wikifin_rag.db_utils import embedding_factory
from wikifin_rag.embedder import Embedder
from hyperopt.pyll import scope

In [26]:
documents = load_wikifin_data(embedding_factory)
embeddings = [doc.pop('embedding') for doc in documents]

In [27]:
embedder = Embedder()

In [28]:
def vector_search(index, query, num_results=5):
    query_vector = embedder.encode(query)
    return index.search(
        query_vector,
        num_results=num_results
    )

In [29]:
def vs_objective(params):
    vs_index = build_vector_index(embeddings, documents, **params)

    metrics = evaluate(
        ground_truth=ground_truth,
        search_function=lambda query: vector_search(index=vs_index, query=query)
    )

    return {
        'loss': -round(metrics['mrr'], 4),
        'status': STATUS_OK,
        'params': params,
        'metrics': metrics
    }

In [30]:
mode_options = ["hnsw", "lsh", "ivf"]
m_options = [8, 12, 16, 24, 32]
ef_construction_options = [50, 100, 200, 400]
ef_search_options = [10, 20, 40, 80, 160, 320]

n_clusters_options = [None, 32, 64, 128, 256, 512]


In [31]:
vs_search_space = hp.pchoice(
    "mode",
    [
        # HNSW
        (0.8, {
            "mode": "hnsw",
            "m": hp.choice("m", [8, 12, 16, 24, 32]),
            "ef_construction": hp.choice(
                "ef_construction",
                [50, 100, 200, 400],
            ),
            "ef_search": hp.choice(
                "ef_search",
                [10, 20, 40, 80, 160, 320],
            )
        }),

        # LSH
        (0.1, {
            "mode": "lsh",
            "n_tables": scope.int(hp.quniform("lsh_n_tables", 2, 32, 1)),
            "hash_size": scope.int(hp.quniform("lsh_hash_size", 8, 24, 1)),
            "n_probe": scope.int(hp.quniform("lsh_n_probe", 0, 2, 1)),
        }),

        # IVF
        (0.1, {
            "mode": "ivf",
            "n_clusters": hp.choice(
                "n_clusters",
                [None, 32, 64, 128, 256, 512]
            ),
            "n_probe_clusters": scope.int(hp.quniform(
                "n_probe_clusters", 1, 32, 1
            )),
        }),
    ]
)

In [32]:
vs_trials_file_path = PROJECT_ROOT / "data" / "evals" / "vs_trials.pkl"

In [33]:
if refresh_search_optimization or not os.path.exists(vs_trials_file_path):
    vs_trials_file_path.parent.mkdir(parents=True, exist_ok=True)
    
    trials = Trials()

    best_params = fmin(
        fn=vs_objective,
        space=vs_search_space,
        algo=tpe.suggest,
        max_evals=n_evals,
        trials=trials
    )

    trials_data = [{
        'id': trial['tid'],
        'mrr': trial['result']['metrics']['mrr'],
        'hit_rate': trial['result']['metrics']['hit_rate'],
        'params': trial['result']['params']
    } for trial in trials.trials]
    trials_data_df = pd.DataFrame(trials_data)

    with open(vs_trials_file_path, "wb") as file:
        pickle.dump(trials_data, file)

  0%|          | 0/25 [00:00<?, ?trial/s, best loss=?]

  0%|          | 0/1500 [00:00<?, ?it/s]

  4%|▍         | 1/25 [00:16<06:45, 16.88s/trial, best loss: -0.5886]

  0%|          | 0/1500 [00:00<?, ?it/s]

  8%|▊         | 2/25 [00:26<04:50, 12.65s/trial, best loss: -0.5886]

...

 92%|█████████▏| 23/25 [1:13:34<24:38, 739.32s/trial, best loss: -0.6233]

  0%|          | 0/1500 [00:00<?, ?it/s]

 96%|█████████▌| 24/25 [1:13:46<08:40, 520.93s/trial, best loss: -0.6233]

  0%|          | 0/1500 [00:00<?, ?it/s]

100%|██████████| 25/25 [1:31:09<00:00, 218.79s/trial, best loss: -0.6233]


In [ ]:
with open(vs_trials_file_path, "rb") as file:
    vs_trials_data = pickle.load(file)

In [60]:
vs_best_metrics = max(vs_trials_data, key=lambda d: d['mrr'])

In [69]:
print(f"Vector search MRR: {round(vs_best_metrics['mrr'], 4)} | Hit Rate: {round(100 * vs_best_metrics['hit_rate'], 2)}%")

Vector search MRR: 0.6233 | Hit Rate: 78.67%


### Hybrid search

In [36]:
vs_best_params = max(vs_trials_data, key=lambda d: d['mrr'])['params']
ts_best_params = max(ts_trials_data, key=lambda d: d['mrr'])['params']

In [37]:
def compute_rrf(rank, k=60):
    return 1 / (k + rank)

In [38]:
def text_search(query, num_results=5):
    return ts_index.search(query=query, boost_dict=ts_best_params, num_results=num_results)

In [44]:
vs_index = build_vector_index(embeddings, documents, **vs_best_params)

In [45]:
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vs_index.search(query_vector=query_vector, num_results=num_results)

In [49]:
def elastic_search_hybrid_rrf(text_search, vector_search, query, num_results=5, k=60):
    ts_results = text_search(query=query, num_results=num_results)
    vs_results = vector_search(query=query, num_results=num_results)

    scores = {}
    doc_map = {}

    for i in range(num_results):
        ts_doc = ts_results[i]
        ts_key = ts_doc["id"]
        doc_map[ts_key] = ts_doc
        scores[ts_key] = scores.get(ts_key, 0) + compute_rrf(i, k)

        vs_doc = vs_results[i]
        vs_key = vs_doc["id"]
        doc_map[vs_key] = vs_doc
        scores[vs_key] = scores.get(vs_key, 0) + compute_rrf(i, k)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in ranked[:num_results]]

In [50]:
def hybrid_search(query):
    return elastic_search_hybrid_rrf(text_search, vector_search, query)

In [ ]:
hs_metrics = evaluate(ground_truth=ground_truth, search_function=hybrid_search)

  0%|          | 0/1500 [00:00<?, ?it/s]

{'hit_rate': 0.776, 'mrr': 0.5916000000000006}

In [ ]:
print(f"Vector search MRR: {round(hs_metrics['mrr'], 4)} | Hit Rate: {round(100 * hs_metrics['hit_rate'], 2)}%")

In [ ]:
openai_client.close()